# 06 — Payload Trade Study

Enumerate valid swap-bay payload configurations, filter by mass and
power budget limits, and rank by endurance impact.


In [ ]:
"""Add the repo root to sys.path so `import models` works from any subdir."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

FIG_DIR = REPO_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from itertools import combinations

with open(REPO_ROOT / "payload" / "payload_manifest.yaml") as f:
    manifest = yaml.safe_load(f)

payloads = manifest["payloads"]
limits = manifest["limits"]
always_on = {n: s for n, s in payloads.items() if s.get("always_on")}
swappable = {n: s for n, s in payloads.items() if not s.get("always_on")}

print("Always-on:", list(always_on))
print("Swappable:", list(swappable))
print("Limits:   ", limits)


## Enumerate valid configurations


In [ ]:
import pandas as pd

base_mass_g  = sum(s["mass_g"]  for s in always_on.values())
base_power_w = sum(s["power_w"] for s in always_on.values())

rows = []
max_n = limits["max_simultaneous_swap_bays"]
for k in range(0, max_n + 1):
    for combo in combinations(swappable.keys(), k):
        mass_g  = base_mass_g  + sum(swappable[n]["mass_g"]  for n in combo)
        power_w = base_power_w + sum(swappable[n]["power_w"] for n in combo)
        valid = (mass_g <= limits["max_payload_mass_g"]
                 and power_w <= limits["max_payload_power_w"])
        rows.append({
            "swap_bays": ", ".join(combo) if combo else "(none)",
            "n_bays": k,
            "total_mass_g": mass_g,
            "total_power_w": power_w,
            "valid": valid,
        })

configs = pd.DataFrame(rows).sort_values(["valid", "total_mass_g"],
                                         ascending=[False, True])
configs.reset_index(drop=True, inplace=True)
configs


## Endurance impact per configuration

Rough first-cut: assume hover current scales linearly with payload
weight via a configuration-independent disc-loading model. Refine by
feeding each config back through notebooks 03/05 once shortlisted.


In [ ]:
from models import propulsion as prop, endurance as ed

# Baseline (light) operating point — replace with NB03/NB05 values.
baseline_auw_kg = 6.5
baseline_hover_current_a = 30.0
pack_voltage_v = 22.2
prop_diameter_in = 18
n_motors = 6
rho = prop.isa_density(500)
A_per_motor = prop.disc_area_m2(prop_diameter_in, 1)

def estimate_hover_current(extra_payload_g: float) -> float:
    auw_kg = baseline_auw_kg + extra_payload_g / 1000.0
    T_per = prop.hover_thrust_per_motor(auw_kg, n_motors)
    P_total = prop.prop_power_estimate(T_per, A_per_motor, rho) * n_motors
    return P_total / pack_voltage_v / 0.85  # ~85 % motor+ESC efficiency

baseline_payload_g = base_mass_g  # always-on only
configs["hover_current_a"] = configs["total_mass_g"].apply(
    lambda m: estimate_hover_current(m - baseline_payload_g) + baseline_hover_current_a
              - estimate_hover_current(0)
)

configs["endurance_min_16ah"] = configs["hover_current_a"].apply(
    lambda i: ed.hover_endurance_min(ed.Battery(capacity_ah=16.0), i)
)
configs.loc[configs["valid"]].sort_values("endurance_min_16ah", ascending=False)
